## Standard Error Validation:

Validates `compute_weighted_se_pv()` against values from PISA data explorer.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from src.data_loader import query_pisa
from src.pisa_stats import compute_weighted_se_pv, weighted_mean_pv

### 1. Load PISA data

In [3]:
COUNTRIES = ["AUS", "CAN", "DEU", "USA"]
YEARS = [2015, 2018, 2022]
SUBJECT = "MATH"

In [4]:
PV_COLS = [f"PV{i}MATH" for i in range(1, 11)]
REP_COLS = [f"W_FSTURWT{r}" for r in range(1, 81)]
BASE_COLS = ["CNT", "YEAR", "OECD", "W_FSTUWT"]

In [5]:
print("Loading data...")
df_raw = query_pisa(
    COUNTRIES,
    year=None,          
    cols=BASE_COLS + PV_COLS + REP_COLS
)

Loading data...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
df_raw = df_raw[df_raw["YEAR"].isin(YEARS)].copy()
print(f"Loaded {len(df_raw):,} rows across {df_raw['CNT'].nunique()} countries "
      f"and {sorted(df_raw['YEAR'].unique().tolist())} cycles.")
df_raw.groupby(["CNT", "YEAR"]).size().rename("n_students")

Loaded 141,197 rows across 4 countries and [2015, 2018, 2022] cycles.


CNT  YEAR
AUS  2015    14530
     2018    14273
     2022    13437
CAN  2015    20058
     2018    22653
     2022    23073
DEU  2015     6504
     2018     5451
     2022     6116
USA  2015     5712
     2018     4838
     2022     4552
Name: n_students, dtype: int64

### 2. Compute weighted means and standard errors

In [7]:
records = []

for cnt in COUNTRIES:
    for year in YEARS:
        subset = df_raw[(df_raw["CNT"] == cnt) & (df_raw["YEAR"] == year)].copy()

        if len(subset) < 30:
            print(f"  Skipping {cnt} {year} - fewer than 30 rows.")
            continue

        mean_score = weighted_mean_pv(subset, SUBJECT)
        se = compute_weighted_se_pv(subset, SUBJECT)

        records.append({
            "CNT": cnt,
            "YEAR": year,
            "calc_mean": round(mean_score, 2),
            "calc_se": round(se, 4),
        })
        print(f"  {cnt} {year}:  mean = {mean_score:.2f}   SE = {se:.4f}")

  AUS 2015:  mean = 493.90   SE = 1.6053
  AUS 2018:  mean = 491.36   SE = 1.9398
  AUS 2022:  mean = 487.08   SE = 1.7796
  CAN 2015:  mean = 515.65   SE = 2.3131
  CAN 2018:  mean = 512.02   SE = 2.3575
  CAN 2022:  mean = 496.95   SE = 1.5618
  DEU 2015:  mean = 505.97   SE = 2.8879
  DEU 2018:  mean = 500.04   SE = 2.6471
  DEU 2022:  mean = 474.83   SE = 3.0648
  USA 2015:  mean = 469.63   SE = 3.1664
  USA 2018:  mean = 478.24   SE = 3.2354
  USA 2022:  mean = 464.89   SE = 4.0067
